<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Exercises_MCP_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises — MCP Client with an LLM

Completed Colab-ready notebook using **MCP over STDIO**.  
The default planner is a deterministic stub, so no API token is required.


In [ ]:
!pip install -q "mcp==1.21.2" requests

In [ ]:
import os

USE_REAL_LLM = False  # Keep False unless GITHUB_TOKEN is configured.
print("Planner mode:", "real LLM" if USE_REAL_LLM else "stub")

Planner mode: stub


## MCP server

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

@mcp.resource("info://server")
def server_info() -> str:
    """Return basic information about the demo MCP server."""
    return "Demo MCP server exposing add, greet, and multiply tools."

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

if __name__ == "__main__":
    mcp.run()


Writing server.py


## Exercise 1 — Theory

**Why is STDIO transport simpler for local MCP development than HTTP?**

STDIO is simpler because the client launches the MCP server as a local child process and exchanges JSON-RPC messages through the process's standard input and output streams. No web server, port, URL routing, CORS configuration, TLS certificate, or network authentication is required. The client also controls the server lifecycle directly.

HTTP is more suitable when a server must be remote or shared by several clients, but it introduces additional networking, deployment, and security configuration.


## Exercise 2 — Connect and initialize

In [ ]:
%%writefile exercise2_client.py
import asyncio
import os
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    with open(os.devnull, "w") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                initialization = await session.initialize()
                print("Exercise 2: OK")
                print("Connected and initialized:", initialization is not None)

if __name__ == "__main__":
    asyncio.run(main())


Writing exercise2_client.py


In [ ]:
!python exercise2_client.py

Exercise 2: OK
Connected and initialized: True


## Exercise 3 — Discover resources and tools

In [ ]:
%%writefile exercise3_client.py
import asyncio
import os
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    with open(os.devnull, "w") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                resources = await session.list_resources()
                print("RESOURCES:")
                for resource in resources.resources:
                    print(f"- name={resource.name}, uri={resource.uri}, description={resource.description}")
                tools = await session.list_tools()
                print("\nTOOLS AND INPUT PROPERTIES:")
                for tool in tools.tools:
                    properties = tool.inputSchema.get("properties", {})
                    print(f"- {tool.name}: {properties}")

if __name__ == "__main__":
    asyncio.run(main())


Writing exercise3_client.py


In [ ]:
!python exercise3_client.py

RESOURCES:
- name=server_info, uri=info://server, description=Return basic information about the demo MCP server.

TOOLS AND INPUT PROPERTIES:
- add: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
- greet: {'name': {'title': 'Name', 'type': 'string'}}
- multiply: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4 — Convert MCP tools to LLM function specifications

The MCP client obtains tool metadata with `session.list_tools()`. Each MCP tool provides a `name`, a `description`, and an `inputSchema` written as JSON Schema.

`convert_to_llm_tool` wraps those fields in the function-calling structure expected by an LLM. The tool name and description tell the model what the function does, while `properties` and `required` define the arguments it may generate. After the LLM proposes a tool call, the MCP client executes it with `session.call_tool()`.


In [ ]:
def convert_to_llm_tool(tool):
    """Convert an MCP Tool object into an LLM function/tool specification."""
    input_schema = tool.inputSchema or {}

    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": input_schema.get("properties", {}),
                "required": input_schema.get("required", []),
            },
        },
    }

## Exercise 5 — Plan and execute

The stub planner reads the prompt, selects an available MCP tool, and creates a function call. The executor then sends that call to the MCP server and prints the returned content.


In [ ]:
%%writefile exercise5_client.py
import asyncio
import json
import os
import re
import sys
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

USE_REAL_LLM = os.getenv("USE_REAL_LLM", "false").lower() == "true"

def convert_to_llm_tool(tool) -> Dict[str, Any]:
    input_schema = tool.inputSchema or {}
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": input_schema.get("properties", {}),
                "required": input_schema.get("required", []),
            },
        },
    }

def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    available_tools = {
        item["function"]["name"]
        for item in functions
        if item.get("type") == "function"
    }
    prompt_lower = prompt.lower()
    numbers = [int(value) for value in re.findall(r"-?\d+", prompt)]
    if ("multiply" in prompt_lower or "product" in prompt_lower) and len(numbers) >= 2:
        if "multiply" not in available_tools:
            raise ValueError("The multiply tool is not available.")
        return [{"name": "multiply", "args": {"a": numbers[0], "b": numbers[1]}}]
    if ("add" in prompt_lower or "sum" in prompt_lower or "+" in prompt) and len(numbers) >= 2:
        if "add" not in available_tools:
            raise ValueError("The add tool is not available.")
        return [{"name": "add", "args": {"a": numbers[0], "b": numbers[1]}}]
    if "greet" in prompt_lower or "hello" in prompt_lower:
        if "greet" not in available_tools:
            raise ValueError("The greet tool is not available.")
        match = re.search(r"(?:greet|hello)\s+([A-Za-z][A-Za-z'-]*)", prompt, re.IGNORECASE)
        name = match.group(1) if match else "Student"
        return [{"name": "greet", "args": {"name": name}}]
    raise ValueError(f"The stub planner cannot create a tool call for: {prompt!r}")

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or keep USE_REAL_LLM=false.")
    try:
        from azure.ai.inference import ChatCompletionsClient
        from azure.core.credentials import AzureKeyCredential
    except ImportError as exc:
        raise RuntimeError("Install azure-ai-inference and azure-core for real LLM mode.") from exc
    client = ChatCompletionsClient(
        endpoint="https://models.inference.ai.azure.com",
        credential=AzureKeyCredential(token),
    )
    response = client.complete(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Plan MCP tool calls."},
            {"role": "user", "content": prompt},
        ],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    message = response.choices[0].message
    for tool_call in message.tool_calls or []:
        arguments = tool_call.function.arguments
        parsed_arguments = json.loads(arguments) if isinstance(arguments, str) else arguments
        calls.append({"name": tool_call.function.name, "args": parsed_arguments})
    return calls

async def main(prompt: str):
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    with open(os.devnull, "w") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                functions = [convert_to_llm_tool(tool) for tool in tools.tools]
                calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                print("prompt:", prompt)
                print("tool_calls:", calls)
                for call in calls:
                    result = await session.call_tool(call["name"], arguments=call["args"])
                    output = [getattr(content, "text", str(content)) for content in result.content]
                    print(f"result from {call['name']}:", output)

if __name__ == "__main__":
    user_prompt = " ".join(sys.argv[1:]) or "Add 2 to 20"
    asyncio.run(main(user_prompt))


Writing exercise5_client.py


In [ ]:
!python exercise5_client.py "Add 2 to 20"

prompt: Add 2 to 20
tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result from add: ['22']


## Optional — Multiply tool

In [ ]:
!python exercise5_client.py "Multiply 6 by 7"

prompt: Multiply 6 by 7
tool_calls: [{'name': 'multiply', 'args': {'a': 6, 'b': 7}}]
result from multiply: ['42']


## Submission checklist

- Exercise 1 theory answer completed.
- MCP server starts locally over STDIO.
- `ClientSession` connects and initializes.
- Resources and tools are listed.
- Tool names and `inputSchema` properties are printed.
- `convert_to_llm_tool` is implemented.
- The stub planner proposes `add(a=2, b=20)`.
- The executor calls the MCP tool and prints `22`.
- The optional `multiply(a=6, b=7)` tool prints `42`.
- `USE_REAL_LLM` remains disabled unless a valid `GITHUB_TOKEN` is configured.
